# Develope pytorch lightning compatible version of measure activations


The following tests each element of `fmri_analysis/measure_layer_activations_165_natural_sounds_lightning.py` block by block 

In [1]:
from __future__ import division
from scipy.io import wavfile
import os

# make sure we are using the correct plotting display. 
import matplotlib 
matplotlib.use('agg')
import matplotlib.pyplot as plt
import numpy as np

import sys
if sys.version_info < (3,):
    from StringIO import StringIO as BytesIO
else:
    from io import BytesIO
import base64

import scipy
import pickle
import h5py
from argparse import ArgumentParser
import pathlib
import yaml

import torch
from lightning_scripts.lightning_ssl_matched_speech_in_noise import LitAudioSSL as LitAudioSSLMatched
import robustness.audio_functions.audio_transforms as at
from robustness.tools.audio_helpers import load_audio_wav_resample
# from analysis_scripts.default_paths import fMRI_DATA_PATH

import itertools
from easydict import EasyDict ## to simulate argparse 


from IPython.display import Audio, display

In [30]:
transforms = at.AudioCompose([
                    at.AudioToTensor(),
                    at.DBSPLNormalizeForegroundAndBackground(60),
                    at.UnsqueezeAudio(dim=0), # dim=0 (Batch, Time)
                    at.UnsqueezeAudio(dim=0) # dim=0 (Batch, 1, Time)
                ])


fMRI_DATA_PATH = pathlib.Path("assets/fMRI_natsound_data")


In [3]:
args = EasyDict()
args.config_path = "model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml"
args.ckpt_path = "model_checkpoints/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment/checkpoints/epoch=160-step=28980-best_word_task.ckpt"
args.model_ckpt_dir = pathlib.Path("./model_checkpoints")
args.save_features_dir = pathlib.Path("./fmri_analysis_model_features")

In [4]:
if args.config_path != "":
    config_path = pathlib.Path(args.config_path)
elif args.config_list_path != "":
    with open(args.config_list_path, 'rb') as f:
        config_dict = pickle.load(f)
        config_path = pathlib.Path(config_dict[args.array_ix])

print(f"Evaluating config: {config_path}")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)


Evaluating config: model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml


In [5]:
if args.ckpt_path == "":
    checkpoint_dir = pathlib.Path(args.model_ckpt_dir) / f"{config_path.stem}/checkpoints"
    ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)
    ckpt_path = ckpt_paths[-1] # get latest checkpoint 
else:
    ckpt_path = args.ckpt_path
print(f"Features from checkpoint: {config_path}")


Features from checkpoint: model_configs/barlow_word_kell2018_base_Matched_blocked_batches_lmbda_1e-2_lr_2e-1_w_augment.yaml


In [6]:
module = LitAudioSSLMatched.load_from_checkpoint(checkpoint_path=ckpt_path, config=config)
all_layers = module.metamer_layers
model = module.model.eval().cuda()

In [7]:
save_features_dir = pathlib.Path(args.save_features_dir)
save_features_dir = save_features_dir / config_path.stem
if not save_features_dir.is_dir():
    save_features_dir.mkdir(parents=True, exist_ok=True)


In [9]:
sound_list = np.load(os.path.join(fMRI_DATA_PATH, 'neural_stim_meta.npy'))

wavs_location = os.path.join(fMRI_DATA_PATH, '165_natural_sounds')

SR=20000 # Match with the networks we are building/training
MEASURE_DUR=2
wav_array = np.empty([165, SR*MEASURE_DUR])
for wav_idx, wav_data in enumerate(sound_list):
    test_audio, SR = load_audio_wav_resample(os.path.join(wavs_location, wav_data[0].decode('utf-8')), DUR_SECS=MEASURE_DUR, resample_SR=SR)
    wav_array[wav_idx,:] = test_audio/np.sqrt(np.mean(test_audio**2))


### Breaking from the .py file to make sure the sounds work

In [31]:
## demo process sound 
sound_idx = -1
sound, _ = transforms(wav_array[sound_idx,:], None)
sound = sound.float().cuda()


In [32]:
sound.shape

torch.Size([1, 1, 40000])

In [34]:
Audio(sound.squeeze().cpu(), rate=SR, normalize=False)

In [36]:
with torch.no_grad():
    predictions, rep, layer_returns = model(sound, with_latent=True) # Corresponding representation


#### Resuming the .py file at line 130

In [40]:
filename = 'natsound_activations'
# only use the non-fake layers
all_layers = [e.split('_fake')[0] for e in all_layers] # Don't duplicate these since we aren't synthesizing
new_all_layers = []
for l_unique in all_layers:
    if l_unique not in new_all_layers:
        new_all_layers.append(l_unique)
all_layers = new_all_layers
net_layer_dict = {}
net_layer_dict_full = {}
net_h5py_file = h5py.File(os.path.join(save_features_dir, filename + '.h5'), "w")
net_h5py_file_full = h5py.File(os.path.join(save_features_dir, filename + '_full.h5'), "w")

# Save the list of layers to the hdf5
net_h5py_file['layer_list'] = np.array([layer.encode("utf-8") for layer in all_layers])
net_h5py_file_full['layer_list'] = np.array([layer.encode("utf-8") for layer in all_layers])


In [43]:
from tqdm import tqdm # adding here for timing, won't be in actual .py

In [ ]:
for sound_idx, sound_info in enumerate(tqdm(sound_list)): # tqdm won't be in actual .py
    ## Could probably process all sounds at once...
    sound, _ = transforms(wav_array[sound_idx,:], None)
    sound = sound.float().cuda()

    with torch.no_grad():
        predictions, rep, layer_returns = model(sound, with_latent=True) # Corresponding representation

    # Make the array have the correct size
    if sound_idx == 0:
        for layer in all_layers:
            print(layer)
            layer_shape_165 = layer_returns[layer].shape
            layer_shape_full = np.prod(np.array(layer_shape_165))
            if len(layer_shape_165)==4:
                layer_shape_unraveled = layer_shape_165[1]*layer_shape_165[2]# don't take the time dimension into account
            else:
                layer_shape_unraveled = layer_shape_165[1]
            net_layer_dict_full[layer] = net_h5py_file_full.create_dataset(layer, (165, layer_shape_full), dtype='float32')
            net_layer_dict[layer] = net_h5py_file.create_dataset(layer, (165, layer_shape_unraveled), dtype='float32')

    for layer_idx, layer in enumerate(all_layers):
        # time averaged features, so that they can be related to the fMRI activations
        if layer_returns[layer].ndim==4: # NCHW (W is time)
            net_layer_dict[layer][sound_idx,:] = np.mean(layer_returns[layer].cpu().detach().numpy(),3).ravel()
        else: # fully connected layers do not have a temporal component.  
            net_layer_dict[layer][sound_idx,:] = layer_returns[layer].cpu().detach().numpy().ravel()
        net_layer_dict_full[layer][sound_idx,:] = layer_returns[layer].cpu().detach().numpy().ravel()
net_h5py_file.close()
net_h5py_file_full.close()

In [45]:
time_avg_fname = os.path.join(save_features_dir, filename + '.h5')

In [47]:
with h5py.File(time_avg_fname, 'r') as h5file:
    print(h5file.keys())
    for key in h5file.keys():
        print(f"{key}, {h5file[key].shape}")

<KeysViewHDF5 ['avgpool', 'batchnorm0', 'batchnorm1', 'batchnorm2', 'conv0', 'conv1', 'conv2', 'conv3', 'conv4', 'dropout', 'fullyconnected', 'input_after_preproc', 'layer_list', 'maxpool0', 'maxpool1', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc']>
avgpool, (165, 2560)
batchnorm0, (165, 211)
batchnorm1, (165, 3456)
batchnorm2, (165, 2304)
conv0, (165, 6816)
conv1, (165, 4608)
conv2, (165, 4608)
conv3, (165, 9216)
conv4, (165, 4608)
dropout, (165, 4096)
fullyconnected, (165, 4096)
input_after_preproc, (165, 211)
layer_list, (20,)
maxpool0, (165, 3456)
maxpool1, (165, 2304)
relu0, (165, 6816)
relu1, (165, 4608)
relu2, (165, 4608)
relu3, (165, 9216)
relu4, (165, 4608)
relufc, (165, 4096)
